In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
ravelry_file_path = r"C:\School\Senior Year (2025-2026)\DAT 490\Datafiles\fixed_final_data.csv"

ravelry_df = pd.read_csv(ravelry_file_path, low_memory = False)
ravelry_df = ravelry_df.sample(n = 500000, random_state = 42)

In [ ]:
print(ravelry_df.columns.tolist())

In [ ]:
unecessaryColumns = [
    "Unnamed: 0.1",
    "permalink",
    "first_photo.id",
    "first_photo.sort_order",
    "first_photo.user_id",
    "first_photo.x_offset",
    "first_photo.y_offset",
    "first_photo.square_url",
    "first_photo.medium_url",
    "first_photo.thumbnail_url",
    "first_photo.small_url",
    "first_photo.medium2_url",
    "first_photo.small2_url",
    "first_photo.caption",
    "first_photo.caption_html",
    "first_photo.copyright_holder",
    "first_photo.aspect_ratio",
    "user.tiny_photo_url",
    "user.small_photo_url",
    "photos_count",
    "user.photo_url",
    "links.self.href",
    "pattern_name",
    "craft_name",
    "status_name"
]

notEnoughData = [
    "Unnamed: 0",
    "first_photo",
    "personal_attributes",
    "ends_per_inch",
    "picks_per_inch",
    "gauge_repeats"
]



ravelry_df = ravelry_df.drop(columns=unecessaryColumns, errors="ignore")
ravelry_df = ravelry_df.drop(columns=notEnoughData, errors="ignore")

In [ ]:
ravelry_df.info()

In [ ]:
df = ravelry_df.copy()

# make date features
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df["created_year"] = df["created_at"].dt.year
df["created_month"] = df["created_at"].dt.month
df["created_quarter"] = df["created_at"].dt.quarter

# target
y = df["favorites_count"].fillna(0)

# predictors
features = [
    "comments_count",
    "rating",
    "progress",
    "project_status_id",
    "completed",
    "started_day_set",
    "completed_day_set",
    "created_year",
    "created_month",
    "created_quarter",
]

X = df[features].copy()

numeric_features = [
    "comments_count",
    "rating",
    "progress",
    "project_status_id",
    "started_day_set",
    "completed_day_set",
    "created_year",
    "created_month",
    "created_quarter",
]


In [ ]:
numeric_transformer = SimpleImputer(strategy="median")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ]
)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=5
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

In [ ]:
# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# fit
model.fit(X_train, y_train)

In [ ]:
# predict
y_pred = model.predict(X_test)

# evaluate
print("R^2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Favorites")
plt.ylabel("Predicted Favorites")
plt.title("Random Forest: Predicted vs Actual")
plt.show()

In [ ]:
# feature names

importances = pd.DataFrame({
    "Feature": numeric_features,
    "Importance": model.named_steps["rf"].feature_importances_
}).sort_values(by="Importance", ascending=False)

top_features = importances.head(10)

plt.figure(figsize=(10, 6))
plt.bar(top_features["Feature"], top_features["Importance"])
plt.xticks(rotation=45, ha="right")
plt.title("Random Forest Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# numeric columns
numeric_df = ravelry_df.select_dtypes(include="number")

# correlation matrix
corr_matrix = numeric_df.corr()

# plot
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()